In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("HotelBookings.csv")

In [ ]:
categorical_cols = ['hotel','meal','country','market_segment','distribution_channel',
                    'reserved_room_type', 'assigned_room_type', 'deposit_type', 'agent'
                    , 'company', 'customer_type']
                    # variables that are chosen to be categorical 

In [ ]:
df[categorical_cols] = df[categorical_cols].astype("category")
# convert chosen variables to categorical type

In [ ]:
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'] + '-' +
    df['arrival_date_day_of_month'].astype(str)
) # form new datetime column by concatenating the data from day, month, year columns

In [ ]:
df = df.drop(columns=[
    'arrival_date_year',
    'arrival_date_month',
    'arrival_date_day_of_month'
]) # drop columns 

In [ ]:
df['room_changed'] = (
    df['reserved_room_type'].astype(str)
    != df['assigned_room_type'].astype(str)
).astype(int)
# form new variable to capture whether reserved and assigned room is different, binary

In [ ]:
df['children'] = df['children'].fillna(0)
# three rows have na values for children, fill with zero(need to give explanation in report probably)

In [ ]:
df['agent_booked'] = df['agent'].notna().astype(int)
# new variable whther an agent booked or not, binary

In [ ]:
df[df['adr']==0] # investigating adr values of 0 shown from profile report

In [ ]:
pd.crosstab(df['is_canceled'], df['adr'].eq(0)) # adr is 0, doesnt mean booking is cancelled

In [ ]:
# 0 values for adr, stays_in_weekend_nights
# and stays_in_week_nights seem suspicious
# flag out the rows

df['zero_night_zero_adr'] = (
    (df['stays_in_weekend_nights'] == 0) &
    (df['stays_in_week_nights'] == 0) &
    (df['adr'] == 0)
)

df['zero_night_zero_adr'].value_counts()
# 517 such rows

In [ ]:

# check no.of guest for possibly problematic rows
df[
    (df['adr'] == 0) &
    (df['stays_in_weekend_nights'] == 0) &
    (df['stays_in_week_nights'] == 0)
][['adults', 'children', 'babies']].value_counts()

# have 0 adults, children and babies seems off

In [ ]:
#investigate records
zero_records = df[
    (df['stays_in_weekend_nights'] == 0) &
    (df['stays_in_week_nights'] == 0) &
    (df['adr'] == 0)
]

pd.crosstab(
    zero_records['adults'],
    zero_records['is_canceled']
)
# There are 46 zero-night/zero-ADR records with no adults, 
# and those may be more suspicious than the 471 records that have 1-3 adults.

In [ ]:
# remove records with 0 adults, 0 children, 0 babies, 0 nights, and 0
# ADR were removed because they contain no recorded guests, stay, or accommodation revenue.
df.shape
df = df[
    ~(
        (df['adults'] == 0) &
        (df['children'] == 0) &
        (df['babies'] == 0) &
        (df['stays_in_weekend_nights'] == 0) &
        (df['stays_in_week_nights'] == 0) &
        (df['adr'] == 0)
    )
]

df.shape

# 46 rows removed

In [ ]:
# from profile report, extreme adr value of 5400, likely misinput
# replace extreme value with median
df.loc[df['adr']==5400, 'adr'] = df['adr'].median

In [10]:
import seaborn as sns

In [ ]:
from ydata_profiling import ProfileReport

# Generate a simple profiling report
profile = ProfileReport(df)

# Display the report in a Jupyter notebook
profile.to_notebook_iframe()

# Save the report to an HTML file
profile.to_file("hotelbookings_profiling_report.html")